# Phase 1 — Problem model and reference instances

Objective: minimize activated hosts.

This notebook validates:
- Input instances.
- Assignment and capacity constraints.
- The objective function.
- Exact reference results for small instances.

In [1]:
import json
import sys
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()

ROOT = next(
    (
        p for p in [cwd, *cwd.parents]
        if (p / "requirements.txt").is_file()
        and (p / "notebooks").is_dir()
    ),
    None,
)

if ROOT is None:
    raise RuntimeError("Cannot locate the project root.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.placement import evaluate_model, evaluate_placement

print("Project root:", ROOT)
print("Python:", sys.executable)

Project root: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement
Python: c:\Users\Khiem\anaconda3\envs\nt409-khiem\python.exe


Tạo bốn bộ dữ liệu tham chiếu

In [2]:
def make_instance(instance_id, tasks, hosts):
    return {
        "schema_version": 1,
        "instance_id": instance_id,
        "objective": "min_active_hosts",
        "resource_units": {
            "cpu": "unit",
            "ram": "unit",
        },
        "tasks": [
            {
                "id": f"T{i + 1}",
                "cpu": cpu,
                "ram": ram,
            }
            for i, (cpu, ram) in enumerate(tasks)
        ],
        "hosts": [
            {
                "id": f"H{h}",
                "cpu": cpu,
                "ram": ram,
            }
            for h, (cpu, ram) in enumerate(hosts)
        ],
    }


fixtures = [
    make_instance(
        "p0_reference",
        tasks=[(2, 4), (3, 2), (1, 3), (4, 5)],
        hosts=[(5, 6), (6, 8)],
    ),
    make_instance(
        "p1_choice",
        tasks=[(1, 1), (1, 2), (2, 1), (1, 1)],
        hosts=[(5, 5), (3, 3)],
    ),
    make_instance(
        "p1_tight",
        tasks=[(2, 1), (2, 1), (1, 2), (1, 2)],
        hosts=[(3, 3), (3, 3)],
    ),
    make_instance(
        "p1_infeasible",
        tasks=[(2, 2), (2, 2), (2, 2), (2, 2)],
        hosts=[(5, 5), (3, 3)],
    ),
]

DATA_DIR = ROOT / "data" / "instances"
DATA_DIR.mkdir(parents=True, exist_ok=True)

for instance in fixtures:
    path = DATA_DIR / f"{instance['instance_id']}.json"

    if not path.exists():
        path.write_text(
            json.dumps(instance, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )

    print(path.name)

instances = {
    item["instance_id"]: json.loads(
        (DATA_DIR / f"{item['instance_id']}.json").read_text(
            encoding="utf-8"
        )
    )
    for item in fixtures
}

p0_reference.json
p1_choice.json
p1_tight.json
p1_infeasible.json


Kiểm tra ràng buộc bằng các trường hợp cụ thể

In [3]:
reference = instances["p0_reference"]

valid = evaluate_placement(reference, [0, 0, 1, 1])

x_bad = np.array(valid["x"])
x_bad[0] = [1, 1]

checks = {
    "valid": valid,
    "overloaded": evaluate_placement(
        reference, [0, 0, 0, 1]
    ),
    "two_hosts_for_T1": evaluate_model(
        reference, x_bad, valid["y"]
    ),
    "H0_disabled": evaluate_model(
        reference, valid["x"], [0, 1]
    ),
}

check_table = pd.DataFrame({
    name: {
        "feasible": result["feasible"],
        "one_hot_ok": result["one_hot_ok"],
        "capacity_ok": result["capacity_ok"],
        "objective": result["objective"],
    }
    for name, result in checks.items()
}).T

display(check_table)

assert checks["valid"]["feasible"]
assert checks["valid"]["objective"] == 2

assert not checks["overloaded"]["feasible"]
assert not checks["two_hosts_for_T1"]["one_hot_ok"]
assert not checks["H0_disabled"]["capacity_ok"]

,feasible,one_hot_ok,capacity_ok,objective
valid,True,True,True,2
overloaded,False,True,False,None
two_hosts_for_T1,False,False,False,None
H0_disabled,False,True,False,None


Kiểm tra mục tiêu tối ưu và tìm đáp án chính xác

In [4]:
choice = instances["p1_choice"]

packed = evaluate_placement(choice, [0, 0, 0, 0])
split = evaluate_placement(choice, [0, 0, 1, 1])

print("Packed:", packed["feasible"], packed["objective"])
print("Split :", split["feasible"], split["objective"])

Packed: True 1
Split : True 2


In [5]:
expected = {
    # feasible_count, best_objective, optimal_count
    "p0_reference": (1, 2, 1),
    "p1_choice": (11, 1, 1),
    "p1_tight": (4, 2, 4),
    "p1_infeasible": (0, None, 0),
}

rows = []

for instance_id, instance in instances.items():
    n = len(instance["tasks"])
    m = len(instance["hosts"])

    feasible_results = []

    for placement in product(range(m), repeat=n):
        result = evaluate_placement(instance, placement)

        if result["feasible"]:
            feasible_results.append(result)

    best = min(
        (r["objective"] for r in feasible_results),
        default=None,
    )

    optimal_results = [
        r for r in feasible_results
        if r["objective"] == best
    ]

    actual = (
        len(feasible_results),
        best,
        len(optimal_results),
    )

    assert actual == expected[instance_id], (
        instance_id, actual
    )

    rows.append({
        "instance_id": instance_id,
        "total_placements": m ** n,
        "feasible_count": len(feasible_results),
        "best_objective": best,
        "optimal_count": len(optimal_results),
    })

summary = pd.DataFrame(rows).set_index("instance_id")
summary["best_objective"] = summary["best_objective"].astype(
    "Int64"
)

display(summary)

,total_placements,feasible_count,best_objective,optimal_count
instance_id,,,,
p0_reference,16,1,2,1
p1_choice,16,11,1,1
p1_tight,16,4,2,4
p1_infeasible,16,0,<NA>,0


Chốt thứ tự biến cho nhánh lượng tử

In [6]:
instance = instances["p0_reference"]
result = evaluate_placement(instance, [0, 0, 1, 1])

n = len(instance["tasks"])
m = len(instance["hosts"])

variable_names = [
    f"x_{i}_{h}"
    for i in range(n)
    for h in range(m)
] + [
    f"y_{h}"
    for h in range(m)
]

model_vector = [
    bit
    for row in result["x"]
    for bit in row
] + result["y"]

display(pd.DataFrame({
    "variable": variable_names,
    "value": model_vector,
}))

print("Number of original binary variables:", len(model_vector))
print("Model vector:", model_vector)

,variable,value
0,x_0_0,1
1,x_0_1,0
2,x_1_0,1
3,x_1_1,0
4,x_2_0,0
5,x_2_1,1
6,x_3_0,0
7,x_3_1,1
8,y_0,1
9,y_1,1


Number of original binary variables: 10
Model vector: [1, 0, 1, 0, 0, 1, 0, 1, 1, 1]
